# S03 - CNN equilibre et prediction d'une nouvelle IRM

Objectif:

1. Charger le train final genere en Partie 2: train equilibre + images augmentees.
2. Entrainer un CNN pour classifier les 4 classes Alzheimer.
3. Evaluer le modele sur le test original.
4. Evaluer aussi la robustesse sur le test augmente, si ce fichier existe.
5. Sauvegarder le modele pour l'application web et l'API.

Classes:

```text
MildDemented
ModerateDemented
NonDemented
VeryMildDemented
```

Important:

Le test original reste la reference principale d'evaluation. Le test augmente sert seulement a verifier la robustesse du modele face a de petites variations d'image. L'objectif est de reduire le biais vers `NonDemented` tout en gardant une lecture detaillee des erreurs par classe.


## 1. Imports et chemins

On charge TensorFlow/Keras pour entrainer un CNN.


In [ ]:
import tensorflow as tf
print(tf.__version__)


In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFilter

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
except Exception as exc:
    raise ImportError(
        "TensorFlow n'est pas installe. Installe-le avec: pip install tensorflow"
    ) from exc

from sklearn.metrics import classification_report, confusion_matrix

plt.style.use("default")
sns.set_theme(style="whitegrid")

PROJECT_DIR = Path("C:/Users/user/Desktop/Alzheimer_CV_Project")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
MODELS_DIR = PROJECT_DIR / "models"
MODELS_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
IMG_SIZE = (160, 160)
BATCH_SIZE = 32
EPOCHS = 20

np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("TensorFlow:", tf.__version__)
print("Processed dir:", PROCESSED_DIR)
print("Models dir:", MODELS_DIR)


### Interpretation

Cette cellule prepare l'environnement d'entrainement.

On utilise une taille `160x160` pour garder un bon compromis entre vitesse et information visuelle. Le modele pourra ensuite etre ameliore avec `224x224` ou du transfer learning.


## 2. Charger les CSV de la Partie 2

On charge les fichiers crees dans la Partie 2:

- `train_final_with_augmentation.csv`: train final = train equilibre + train augmente;
- `test_split.csv`: test original, utilise pour l'evaluation principale;
- `test_augmented_robustness.csv`: test augmente, utilise seulement pour tester la robustesse;
- `class_weights.csv`: poids de classes sauvegardes pour documentation.


In [ ]:
train_final_path = PROCESSED_DIR / "train_final_with_augmentation.csv"
train_augmented_path = PROCESSED_DIR / "train_augmented.csv"
train_balanced_path = PROCESSED_DIR / "train_balanced_oversampled.csv"
test_path = PROCESSED_DIR / "test_split.csv"
test_augmented_path = PROCESSED_DIR / "test_augmented_robustness.csv"
class_weights_path = PROCESSED_DIR / "class_weights.csv"

if train_final_path.exists():
    train_balanced_df = pd.read_csv(train_final_path)
    train_source = "Train final = train equilibre + train augmente"
elif train_balanced_path.exists():
    train_balanced_df = pd.read_csv(train_balanced_path)
    train_source = "Train equilibre seulement"
elif train_augmented_path.exists():
    train_balanced_df = pd.read_csv(train_augmented_path)
    train_source = "Train augmente seulement"
else:
    raise FileNotFoundError("Aucun fichier train disponible. Lance d'abord la Partie 2.")

if not test_path.exists():
    raise FileNotFoundError("test_split.csv introuvable. Lance d'abord la Partie 2.")

test_df = pd.read_csv(test_path)
test_augmented_df = pd.read_csv(test_augmented_path) if test_augmented_path.exists() else None
class_weights_df = pd.read_csv(class_weights_path) if class_weights_path.exists() else None

# Ordre fixe pour garder les memes indices dans le notebook, le modele et l'application.
classes = ["MildDemented", "ModerateDemented", "NonDemented", "VeryMildDemented"]
class_to_index = {class_name: idx for idx, class_name in enumerate(classes)}
index_to_class = {idx: class_name for class_name, idx in class_to_index.items()}

print("Train utilise:", train_source)
print("Train images:", len(train_balanced_df))
print("Test original images:", len(test_df))
print("Test augmente images:", len(test_augmented_df) if test_augmented_df is not None else 0)
print("Classes:", classes)

print("\nDistribution train:")
display(train_balanced_df["label"].value_counts().reindex(classes).fillna(0).astype(int))

print("\nDistribution test original:")
display(test_df["label"].value_counts().reindex(classes).fillna(0).astype(int))

if test_augmented_df is not None:
    print("\nDistribution test augmente robustesse:")
    display(test_augmented_df["label"].value_counts().reindex(classes).fillna(0).astype(int))

if class_weights_df is not None:
    print("\nClass weights sauvegardes en Partie 2:")
    display(class_weights_df)


### Interpretation

Le train utilise maintenant la version finale de la Partie 2 quand elle existe: `train equilibre + train augmente`.

Le test original est garde comme evaluation principale, car il represente les images reelles non modifiees. Le test augmente est separe: il ne sert pas a annoncer la performance principale, mais a verifier si le modele reste stable quand l'image subit de petites transformations.


## 3. Visualiser train final, test original et test augmente

On compare les tailles des ensembles utilises. L'echelle logarithmique est utile car le train final est beaucoup plus grand que le test.


In [ ]:
plot_parts = [
    train_balanced_df.assign(dataset="Train final"),
    test_df.assign(dataset="Test original"),
]
if test_augmented_df is not None:
    plot_parts.append(test_augmented_df.assign(dataset="Test augmente"))

plot_df = pd.concat(plot_parts, ignore_index=True)

count_table = (
    plot_df
    .groupby(["dataset", "label"])
    .size()
    .reset_index(name="n_images")
)

summary_table = count_table.pivot(index="label", columns="dataset", values="n_images").fillna(0).astype(int).reindex(classes)
display(summary_table)

plt.figure(figsize=(12, 5))
ax = sns.barplot(data=count_table, x="label", y="n_images", hue="dataset", order=classes)
plt.title("Distribution utilisee dans la Partie 3")
plt.xlabel("Classe")
plt.ylabel("Nombre d'images")
plt.xticks(rotation=25, ha="right")
for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", fontsize=8, padding=2)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
ax = sns.barplot(data=count_table, x="label", y="n_images", hue="dataset", order=classes)
plt.yscale("log")
plt.title("Distribution avec echelle log - les petits tests deviennent visibles")
plt.xlabel("Classe")
plt.ylabel("Nombre d'images, echelle log")
plt.xticks(rotation=25, ha="right")
for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", fontsize=8, padding=2)
plt.tight_layout()
plt.show()


### Interpretation

Le train final est volontairement plus grand: il contient les images equilibrees et les images augmentees.

Le test original reste plus petit et garde sa distribution naturelle. C'est normal que les barres du test paraissent tres petites face au train final; l'echelle log permet de les voir clairement.


## 4. Fonctions de preprocessing image

On applique le meme preprocessing que dans la Partie 2: crop du fond noir, resize et normalisation.


In [ ]:
def crop_non_black_region_pil(img, threshold=8, padding=6):
    arr = np.array(img)
    mask = arr > threshold
    if mask.sum() == 0:
        return img

    ys, xs = np.where(mask)
    y0, y1 = ys.min(), ys.max()
    x0, x1 = xs.min(), xs.max()

    y0 = max(0, y0 - padding)
    y1 = min(arr.shape[0] - 1, y1 + padding)
    x0 = max(0, x0 - padding)
    x1 = min(arr.shape[1] - 1, x1 + padding)
    return img.crop((x0, y0, x1 + 1, y1 + 1))


def load_preprocess_image(path, img_size=IMG_SIZE):
    img = Image.open(path).convert("L")
    img = crop_non_black_region_pil(img)
    img = img.filter(ImageFilter.MedianFilter(size=3))
    img = img.resize(img_size, Image.Resampling.BILINEAR)
    arr = np.array(img, dtype=np.float32) / 255.0
    arr = np.expand_dims(arr, axis=-1)
    return arr


def dataframe_to_arrays(dataframe):
    missing = [path for path in dataframe["image_path"] if not Path(path).exists()]
    if missing:
        raise FileNotFoundError(f"Images introuvables, exemple: {missing[0]}")
    X = np.stack([load_preprocess_image(path) for path in dataframe["image_path"]])
    y = dataframe["label"].map(class_to_index).values.astype(np.int64)
    return X, y

print("Preprocessing ready: crop + denoising leger + resize + normalisation.")


### Interpretation

Le preprocessing est maintenant coherent avec l'application finale:

1. crop du fond noir;
2. denoising leger par filtre median;
3. redimensionnement;
4. normalisation entre 0 et 1.

On n'applique pas de contraste fort, car cela peut modifier artificiellement l'apparence des IRM.


## 5. Charger les images en tableaux numpy

Cette cellule peut prendre un peu de temps car elle lit toutes les images du train, test.


In [ ]:
X_train, y_train = dataframe_to_arrays(train_balanced_df)
X_test, y_test = dataframe_to_arrays(test_df)

if test_augmented_df is not None:
    X_test_aug, y_test_aug = dataframe_to_arrays(test_augmented_df)
else:
    X_test_aug, y_test_aug = None, None

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)
if X_test_aug is not None:
    print("X_test_aug:", X_test_aug.shape, "y_test_aug:", y_test_aug.shape)


### Interpretation

Les donnees sont maintenant pretes pour le CNN.

Le train est equilibre. Le test sera utilise pour l evaluation finale apres entrainement.


## 6. Visualiser les images apres preprocessing

On verifie que les images restent lisibles apres crop, resize et normalisation.


In [ ]:
plt.figure(figsize=(14, 8))
for i in range(12):
    idx = np.random.randint(0, len(X_train))
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_train[idx].squeeze(), cmap="gray")
    plt.title(index_to_class[int(y_train[idx])], fontsize=10)
    plt.axis("off")

plt.suptitle("Images train apres preprocessing", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()


### Interpretation

Cette verification est importante: le preprocessing ne doit pas couper le cerveau ou supprimer des informations utiles.

Si les images restent bien visibles, on peut passer a l'entrainement du modele.


## 7. Data augmentation pour le CNN

On ajoute une augmentation geometrique raisonnable pendant l'entrainement du CNN.

Important: on ne met pas de contraste aleatoire ici, pour rester coherent avec la decision prise dans la Partie 2.


In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomRotation(0.04),
    layers.RandomZoom(0.08),
    layers.RandomTranslation(0.04, 0.04),
], name="medical_data_augmentation")

plt.figure(figsize=(12, 6))
example = X_train[0:1]
for i in range(8):
    augmented = data_augmentation(example, training=True).numpy()[0]
    plt.subplot(2, 4, i + 1)
    plt.imshow(augmented.squeeze(), cmap="gray")
    plt.axis("off")
plt.suptitle("Exemples d'augmentation geometrique appliquee au train", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()


### Interpretation

L'augmentation aide le modele a generaliser sans creer des IRM artificiellement trop modifiees.

Les transformations sont faibles: petite rotation, petit zoom et petite translation. Cela simule des differences de cadrage ou d'acquisition, mais ne change pas la classe medicale.


## 8. Construire un CNN simple et robuste

On construit un CNN adapte aux images IRM en niveaux de gris.


In [ ]:
def build_cnn_model(input_shape=(160, 160, 1), n_classes=4):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)

    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.35)(x)
    outputs = layers.Dense(n_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name="Balanced_Alzheimer_CNN")
    return model

model = build_cnn_model(input_shape=(*IMG_SIZE, 1), n_classes=len(classes))
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()


### Interpretation

Le CNN apprend automatiquement des filtres visuels:

- contours;
- textures;
- contrastes;
- formes;
- structures internes.

La sortie `softmax` donne une probabilite pour chaque classe Alzheimer.


## 9. Entrainer le modele

On utilise des callbacks pour garder le meilleur modele et eviter le surapprentissage.


In [ ]:
model_path = MODELS_DIR / "balanced_alzheimer_cnn.keras"

callbacks = [
    keras.callbacks.ModelCheckpoint(
        model_path,
        monitor="accuracy",
        save_best_only=True,
        mode="max",
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="loss",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
]

history = model.fit(
    X_train,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)


### Interpretation

Le modele apprend sur le train final augmente.

Comme on a annule la validation separee, les courbes d'entrainement montrent seulement l'apprentissage sur le train. La vraie verification de generalisation se fait ensuite sur `test original`, puis en robustesse sur `test augmente`.


## 10. Courbes d'apprentissage

On visualise accuracy et loss pour detecter le surapprentissage.


In [ ]:
hist = pd.DataFrame(history.history)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(hist["accuracy"], label="train accuracy")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.2)

plt.subplot(1, 2, 2)
plt.plot(hist["loss"], label="train loss")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.2)

plt.tight_layout()
plt.show()


### Interpretation

Ces courbes montrent si le modele apprend correctement sur le train.

Comme il n'y a pas de validation separee dans cette version, il ne faut pas conclure uniquement avec ces courbes. La conclusion principale doit venir du rapport de classification et de la matrice de confusion sur le test original.


## 11. Evaluation finale sur le test

On evalue le meilleur modele sauvegarde sur le test original.


In [ ]:
best_model = keras.models.load_model(model_path)

test_loss, test_acc = best_model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

y_proba = best_model.predict(X_test)
y_pred = np.argmax(y_proba, axis=1)

print(classification_report(y_test, y_pred, target_names=classes))


### Interpretation

Le rapport de classification est plus important que l'accuracy seule.

On doit regarder le F1-score de chaque classe, surtout `ModerateDemented`, car c'est la classe la plus minoritaire.


## 12. Matrice de confusion

On visualise les confusions entre classes.


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
plt.title("Matrice de confusion - Test")
plt.xlabel("Prediction")
plt.ylabel("Vraie classe")
plt.xticks(rotation=25, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


### Interpretation

La matrice de confusion montre quelles classes sont confondues.

Si beaucoup de `MildDemented` ou `VeryMildDemented` sont predits comme `NonDemented`, cela signifie que le modele ne detecte pas assez bien les stades precoces.

L'objectif est de reduire ces confusions grace a l'equilibrage et a l'augmentation.


## 12 bis. Evaluation sur test augmente

Cette evaluation mesure la robustesse du modele sur des images de test legerement modifiees.

Le score principal reste celui du test original. Le test augmente est une analyse complementaire.


In [ ]:
# ============================================================
# EVALUATION SUR TEST AUGMENTE - ROBUSTESSE
# ============================================================

if X_test_aug is not None:
    aug_loss, aug_acc = best_model.evaluate(X_test_aug, y_test_aug, verbose=0)
    print(f"Augmented test loss: {aug_loss:.4f}")
    print(f"Augmented test accuracy: {aug_acc:.4f}")

    y_proba_aug = best_model.predict(X_test_aug)
    y_pred_aug = np.argmax(y_proba_aug, axis=1)

    print(classification_report(y_test_aug, y_pred_aug, target_names=classes))

    cm_aug = confusion_matrix(y_test_aug, y_pred_aug)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm_aug, annot=True, fmt="d", cmap="Purples", xticklabels=classes, yticklabels=classes)
    plt.title("Matrice de confusion - Test augmente robustesse")
    plt.xlabel("Prediction")
    plt.ylabel("Vraie classe")
    plt.xticks(rotation=25, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("Aucun test augmente disponible. Lance d'abord la Partie 2 pour generer test_augmented_robustness.csv.")


## 13. Visualiser des predictions correctes et incorrectes

On affiche des exemples pour comprendre le comportement du modele.


In [ ]:
test_vis_df = test_df.copy().reset_index(drop=True)
test_vis_df["true_idx"] = y_test
test_vis_df["pred_idx"] = y_pred
test_vis_df["true_label"] = [index_to_class[i] for i in y_test]
test_vis_df["pred_label"] = [index_to_class[i] for i in y_pred]
test_vis_df["confidence"] = y_proba.max(axis=1)
test_vis_df["correct"] = test_vis_df["true_idx"] == test_vis_df["pred_idx"]

def show_prediction_examples(dataframe, title, n=12):
    sample = dataframe.sample(min(n, len(dataframe)), random_state=RANDOM_STATE)
    plt.figure(figsize=(14, 9))
    for i, (_, row) in enumerate(sample.iterrows()):
        img = load_preprocess_image(row["image_path"])
        plt.subplot(3, 4, i + 1)
        plt.imshow(img.squeeze(), cmap="gray")
        color = "green" if row["correct"] else "red"
        plt.title(f"Vrai: {row['true_label']}\nPred: {row['pred_label']} ({row['confidence']:.2f})", color=color, fontsize=9)
        plt.axis("off")
    plt.suptitle(title, fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()

show_prediction_examples(test_vis_df[test_vis_df["correct"]], "Predictions correctes")
show_prediction_examples(test_vis_df[~test_vis_df["correct"]], "Predictions incorrectes")


### Interpretation

Les exemples corrects montrent ce que le modele reconnait bien.

Les erreurs montrent les limites du modele. Les confusions entre `VeryMildDemented` et `NonDemented` sont attendues, car les differences anatomiques sont tres subtiles.


## 14. Fonction de prediction pour une nouvelle image

Cette fonction permet d'injecter n'importe quelle IRM et d'obtenir la classe predite.


In [ ]:
def predict_alzheimer_image(image_path, model=best_model, show=True):
    arr = load_preprocess_image(image_path)
    proba = model.predict(arr[np.newaxis, ...], verbose=0)[0]
    pred_idx = int(np.argmax(proba))
    pred_label = index_to_class[pred_idx]
    confidence = float(proba[pred_idx])

    result_df = pd.DataFrame({
        "class": classes,
        "probability": proba,
    }).sort_values("probability", ascending=False)

    if show:
        plt.figure(figsize=(10, 4))
        plt.subplot(1, 2, 1)
        plt.imshow(arr.squeeze(), cmap="gray")
        plt.title(f"Prediction: {pred_label}\nConfidence: {confidence:.2f}")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        sns.barplot(data=result_df, x="probability", y="class")
        plt.xlim(0, 1)
        plt.title("Probabilites par classe")
        plt.tight_layout()
        plt.show()

    return pred_label, confidence, result_df

# Exemple: tester avec une image du test
example_path = test_df.sample(1, random_state=RANDOM_STATE).iloc[0]["image_path"]
pred_label, confidence, probabilities = predict_alzheimer_image(example_path)
print("Image:", example_path)
print("Prediction:", pred_label)
print("Confidence:", confidence)
display(probabilities)


### Interpretation

Cette fonction sera reutilisee dans l'application finale.

Elle prend une image, applique le meme preprocessing, puis retourne:

- la classe predite;
- la confiance;
- les probabilites pour chaque classe.

C'est la base de l'application frontend/API que nous construirons plus tard.


## 15. Sauvegarder le mapping des classes

On sauvegarde l'ordre des classes pour que l'application interprete correctement les predictions.


In [ ]:
class_mapping_path = MODELS_DIR / "class_mapping.json"
with open(class_mapping_path, "w", encoding="utf-8") as f:
    json.dump({"class_to_index": class_to_index, "index_to_class": index_to_class}, f, indent=2)

print("Model saved:", model_path)
print("Class mapping saved:", class_mapping_path)


### Interpretation

Le fichier `class_mapping.json` garantit que l'application saura quelle sortie du modele correspond a quelle classe.

Sans ce mapping, on pourrait inverser les labels et afficher une mauvaise prediction.


# Conclusion de l'etape 3

Cette etape entraine le CNN avec le train final de la Partie 2: train equilibre + train augmente.

Corrections appliquees dans cette version:

1. chargement correct de `train_final_with_augmentation.csv`;
2. evaluation principale sur `test_split.csv`;
3. evaluation complementaire sur `test_augmented_robustness.csv` si disponible;
4. preprocessing coherent avec l'application: crop + denoising leger + resize + normalisation;
5. suppression du contraste aleatoire dans l'augmentation du CNN;
6. interpretations mises a jour sans parler de validation separee.

Pour juger le modele, il faut surtout regarder:

1. l'accuracy sur le test original;
2. le F1-score par classe;
3. la matrice de confusion;
4. les erreurs sur `ModerateDemented`, car cette classe est tres minoritaire;
5. la robustesse sur le test augmente.

Etape suivante:

- utiliser Grad-CAM pour expliquer les zones regardees par le CNN;
- utiliser l'API FastAPI et l'application web pour tester une nouvelle IRM.
